# Use Case Feasibility Check

This notebook validates whether the documented use cases map to the current implementation surface of the agent. It runs against the real FastAPI app, but forces the LLM into mock mode so the check stays deterministic and cheap.

## What This Notebook Verifies

1. The three audit skills can be executed through the structured audit endpoint.
2. The SOP-to-skill flow can generate a structured draft.
3. The results are grounded in the current API and schema, not only in README prose.

Important caveat: the current approval review flow can flag suspicious approve calls and unknown spenders, but it does not yet decode exact allowance semantics from ABI-level transaction data.

In [ ]:
import json
import os
from copy import deepcopy
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'app').exists():
            return candidate
    raise RuntimeError('Project root not found')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
os.environ['LLM_MODE'] = 'mock'
os.environ['APP_DB_PATH'] = str(PROJECT_ROOT / 'app_data' / 'notebook-agent.db')

from app.config import reset_settings_cache
from app.kb.search import reset_kb_for_tests
from app.skills.registry import reset_registry_for_tests

reset_settings_cache()
reset_registry_for_tests()
reset_kb_for_tests()

from fastapi.testclient import TestClient
from app.main import create_app

client = TestClient(create_app())
PROJECT_ROOT

In [ ]:
base_payload = json.loads((PROJECT_ROOT / 'examples' / 'audit_request.json').read_text(encoding='utf-8'))
expert_text = (PROJECT_ROOT / 'examples' / 'expert_sop.txt').read_text(encoding='utf-8')

def audit_payload(*, skill_id: str, objective: str, transactions=None, balances=None, ledger_entries=None, address_labels=None, knowledge_texts=None, extra_notes=None):
    payload = deepcopy(base_payload)
    payload['skill_id'] = skill_id
    payload['objective'] = objective
    if transactions is not None:
        payload['transactions'] = transactions
    if balances is not None:
        payload['balances'] = balances
    if ledger_entries is not None:
        payload['ledger_entries'] = ledger_entries
    if address_labels is not None:
        payload['address_labels'] = address_labels
    if knowledge_texts is not None:
        payload['knowledge_texts'] = knowledge_texts
    if extra_notes is not None:
        payload['extra_notes'] = extra_notes
    return payload

def post_json(path: str, body: dict) -> dict:
    response = client.post(path, json=body)
    return {
        'status_code': response.status_code,
        'body': response.json(),
    }

In [ ]:
ledger_payload = audit_payload(
    skill_id='ledger_reconciliation_review',
    objective='Reconcile month-end on-chain treasury activity against the internal ledger.'
)

counterparty_payload = audit_payload(
    skill_id='counterparty_risk_review',
    objective='Review unknown counterparties and first-seen outflows in the treasury window.',
    address_labels={},
    extra_notes='Treat unlabeled outgoing counterparties as review targets.'
)

approval_transactions = [
    {
        'tx_hash': '0xapprove1',
        'timestamp': '2026-01-15T16:45:00+00:00',
        'chain': 'ethereum',
        'from_address': '0x1111111111111111111111111111111111111111',
        'to_address': '0xfeedfeedfeedfeedfeedfeedfeedfeedfeedfeed',
        'asset_symbol': 'USDC',
        'amount': 5000,
        'direction': 'out',
        'tx_type': 'approve',
        'notes': 'Unlimited allowance requested: type(uint256).max'
    },
    {
        'tx_hash': '0xapprove2',
        'timestamp': '2026-01-15T17:15:00+00:00',
        'chain': 'ethereum',
        'from_address': '0x1111111111111111111111111111111111111111',
        'to_address': '0xfeedfeedfeedfeedfeedfeedfeedfeedfeedfeed',
        'asset_symbol': 'USDC',
        'amount': 5000,
        'direction': 'out',
        'tx_type': 'approve',
        'notes': 'Repeated approval to the same unknown spender.'
    }
]
approval_payload = audit_payload(
    skill_id='approval_risk_review',
    objective='Review token approval risk before interacting with a new DeFi protocol.',
    transactions=approval_transactions,
    balances=base_payload['balances'][:1],
    ledger_entries=[],
    address_labels={},
    knowledge_texts=['Treasury policy: approvals to unknown spenders require manual review.'],
    extra_notes='This simulates approval risk, but not ABI-level allowance decoding.'
)

skill_draft_payload = {
    'skill_name': 'AML Multi Sig Review',
    'domain': 'blockchain-audit',
    'expert_text': expert_text,
    'notes': 'Notebook validation run.'
}

In [ ]:
results = {
    'use_case_1_ledger': post_json('/v1/audit/run', ledger_payload),
    'use_case_2_counterparty': post_json('/v1/audit/run', counterparty_payload),
    'use_case_3_approval': post_json('/v1/audit/run', approval_payload),
    'use_case_4_sop_to_skill': post_json('/v1/skills/generate', skill_draft_payload),
}

summary_rows = []
for name, result in results.items():
    body = result['body']
    summary_rows.append({
        'use_case': name,
        'status_code': result['status_code'],
        'status': body.get('status', 'n/a'),
        'has_result': 'result' in body or 'draft' in body,
        'keys': sorted(body.keys()),
    })

summary_rows

In [ ]:
feasibility_assessment = [
    {
        'use_case': 'Institutional Ledger Reconciliation',
        'verdict': 'directly supported',
        'reason': 'There is a dedicated audit skill plus normalized ledger, balance, and transaction schemas.'
    },
    {
        'use_case': 'Counterparty Risk Review',
        'verdict': 'directly supported',
        'reason': 'Unknown counterparty, first-seen counterparty, and large outflow heuristics already exist.'
    },
    {
        'use_case': 'Approval Risk Review',
        'verdict': 'partially supported',
        'reason': 'The agent can flag unknown spender approvals and repeated approve calls, but it does not decode exact allowance values from calldata yet.'
    },
    {
        'use_case': 'SOP to Skill Conversion',
        'verdict': 'supported with one extra save step',
        'reason': 'Draft generation is built in, but persistence requires an explicit call to /v1/skills/save after generation.'
    },
]
feasibility_assessment

## Recommended Reading of the Results

- If all status codes are 200 in mock mode, the use case is implementable on the current backend surface.
- For real Vertex runs, separate infrastructure factors still matter: quota, region availability, and ADC setup.
- The approval scenario should be treated as workflow-level approval review, not full calldata or ABI analysis.
- The SOP scenario becomes fully operational only after the generated draft is posted to the save endpoint.